# Ajuste fino con tu proyecto

**Sesión 7 · IA Intermedio · EstudIA 2026**

Hoy haces el experimento de adaptación de tu proyecto: ajustas un modelo con tus propios ejemplos y lo comparas con
los embeddings congelados de la sesión 6, **con los mismos ejemplos y la misma validación cruzada**.

| Camino | Para quién | Qué haces |
|---|---|---|
| **A1. Texto** | Tu proyecto clasifica textos | Ajuste fino de un modelo de texto pequeño frente a embeddings + regresión logística |
| **A2. Imagen** | Tu proyecto clasifica imágenes | Ajuste fino de un modelo de imágenes (la receta de la sesión 4) frente a sus embeddings congelados |
| **B. Práctica** | Todavía no tienes ejemplos | Haces A1 con los 30 mensajes ciudadanos |

Cada resultado queda en tu **registro de experimentos**; al final, el registro arma tu tabla de comparación. Con pocos
ejemplos, puede que el ajuste fino pierda: es un resultado tan válido como cualquier otro, y hay que documentarlo.
Cuando termines, **entrega el notebook** desde la última celda. No importa si el modelo acierta.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. Con pocos ejemplos funciona sin GPU; con GPU va más
rápido.

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, json, zipfile, urllib.request, warnings
from collections import Counter
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s07")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

try:
    from sentence_transformers import SentenceTransformer
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, set_seed, pipeline)
from transformers.utils import logging as registro_hf
registro_hf.set_verbosity_error()   # sin avisos largos al cargar los modelos
try:
    from peft import LoraConfig, get_peft_model
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
    from peft import LoraConfig, get_peft_model

GPU = torch.cuda.is_available()
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Listo.", f"GPU: {torch.cuda.get_device_name(0)}." if GPU else "Sin GPU: alcanza con pocos ejemplos.")

In [ ]:
# @title Funciones auxiliares: números, evaluar, validación cruzada, ajustar y registrar
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

def evaluar(verdad, prediccion, titulo="Tu modelo"):
    """Compara las predicciones con las etiquetas reales y con la línea base de la clase más frecuente."""
    verdad, prediccion = [str(v) for v in verdad], [str(p) for p in prediccion]
    clases = sorted(set(verdad))
    frecuente = max(clases, key=verdad.count)
    def balanceada(p):
        return float(np.mean([np.mean([pi == c for vi, pi in zip(verdad, p) if vi == c]) for c in clases]))
    print(f"{titulo} · {len(verdad)} ejemplos, {len(clases)} clases")
    print(f"  Exactitud:            {pct(np.mean([v == p for v, p in zip(verdad, prediccion)]))}   "
          f"(línea base “siempre {frecuente}”: {pct(np.mean([v == frecuente for v in verdad]))})")
    print(f"  Exactitud balanceada: {pct(balanceada(prediccion))}   (línea base: {pct(balanceada([frecuente] * len(verdad)))})")
    todas = clases + sorted(set(prediccion) - set(clases))
    fig, eje = plt.subplots(figsize=(1.1 * len(todas) + 2.5, 0.9 * len(clases) + 1.8))
    ConfusionMatrixDisplay(confusion_matrix(verdad, prediccion, labels=todas), display_labels=todas).plot(
        ax=eje, cmap="Purples", colorbar=False, xticks_rotation=45)
    eje.set_xlabel("Lo que respondió"); eje.set_ylabel("La etiqueta real"); eje.set_title(titulo)
    eje.set_ylim(len(clases) - 0.5, -0.5)
    plt.tight_layout(); plt.show()

def _pliegues(etiquetas):
    n = int(min(5, np.bincount(np.unique(etiquetas, return_inverse=True)[1]).min()))
    if n < 2:
        print("Necesitas al menos 2 ejemplos de cada etiqueta.")
    return n


def bolsa_con_validacion_cruzada(textos, etiquetas):
    """Bolsa de palabras + regresión logística, medida con validación cruzada (la de la sesión 5)."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    modelo = make_pipeline(TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None, ngram_range=(1, 2)),
                           LogisticRegression(max_iter=3000, C=10, class_weight="balanced"))
    return cross_val_predict(modelo, list(textos), etiquetas, cv=StratifiedKFold(n, shuffle=True, random_state=0))

def embeddings_con_validacion_cruzada(X, etiquetas, C=10):
    """Regresión logística sobre embeddings ya calculados (una fila por ejemplo), con validación cruzada."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    return cross_val_predict(LogisticRegression(max_iter=5000, C=C, class_weight="balanced"), X, etiquetas,
                             cv=StratifiedKFold(n, shuffle=True, random_state=0))

def zero_shot(X, etiquetas_posibles, D):
    """A cada fila de X le da la etiqueta cuya descripción (fila de D) queda más cerca."""
    return [etiquetas_posibles[i] for i in np.argmax(X @ D.T, axis=1)]

def parecidos_con_otra_etiqueta(X, ejemplos, etiquetas, cuantos=5):
    """Los pares de ejemplos más parecidos entre sí que tienen etiquetas distintas: posibles etiquetas dudosas."""
    S = X @ X.T
    pares = [(S[i, j], i, j) for i in range(len(X)) for j in range(i + 1, len(X)) if etiquetas[i] != etiquetas[j]]
    for s, i, j in sorted(pares, reverse=True)[:cuantos]:
        print(f"{s:.2f}".replace(".", ","), f"· [{etiquetas[i]}] {str(ejemplos[i])[:70]}")
        print("     ", f"[{etiquetas[j]}] {str(ejemplos[j])[:70]}")

def ajuste_con_validacion_cruzada(textos, etiquetas, modelo, tasa=1e-4, epocas=10, forma="todo"):
    """Ajuste fino medido con validación cruzada, con los mismos grupos que las otras funciones: cada ejemplo se
    predice con un modelo ajustado sin él."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    textos = np.array([str(t) for t in textos], dtype=object)
    pred = np.empty(len(etiquetas), dtype=object)
    inicio = time.time()
    for i, (tr, te) in enumerate(StratifiedKFold(n, shuffle=True, random_state=0).split(textos, etiquetas)):
        a = ajustar(textos[tr], etiquetas[tr], modelo, forma=forma, tasa=tasa, epocas=epocas, lote=8, mostrar=False)
        pred[te] = predecir(a, textos[te])
        print(f"  grupo {i + 1} de {n} listo ({time.time() - inicio:.0f} s)")
    return pred

def exactitudes(verdad, prediccion):
    """Exactitud y exactitud balanceada, en porcentaje, para el registro."""
    verdad, prediccion = np.array([str(v) for v in verdad]), np.array([str(p) for p in prediccion])
    bal = np.mean([np.mean(prediccion[verdad == c] == c) for c in np.unique(verdad)])
    return round(100 * float(np.mean(verdad == prediccion)), 1), round(100 * float(bal), 1)

class Ajuste:
    """Lo que queda de un ajuste fino: el entrenador, el tokenizador, las clases y unos datos para el registro."""
    def __init__(self, **datos):
        self.__dict__.update(datos)

def _ejemplos(tok, textos, largo, clases=None, etiquetas=None):
    """Convierte los textos en tokens (cortando en `largo`) y, si hay etiquetas, les pone su número de clase."""
    cod = tok([str(t) for t in textos], truncation=True, max_length=largo)
    filas = [{k: cod[k][i] for k in cod} for i in range(len(textos))]
    if etiquetas is not None:
        for fila, e in zip(filas, etiquetas):
            fila["labels"] = clases.index(str(e))
    return filas

def ajustar(textos, etiquetas, modelo, forma="todo", tasa=5e-5, epocas=3, lote=16, largo=128,
            val_textos=None, val_etiquetas=None, semilla=0, mostrar=True):
    """Ajuste fino de un modelo de texto de Hugging Face para clasificar, con el Trainer.

    forma: "todo" (todo el modelo), "cabeza" (solo la capa nueva de encima) o "lora" (matrices pequeñas añadidas).
    Si le das textos de validación, mide la exactitud de validación al final de cada época."""
    set_seed(semilla)
    clases = sorted({str(e) for e in etiquetas})
    tok = AutoTokenizer.from_pretrained(modelo)
    red = AutoModelForSequenceClassification.from_pretrained(
        modelo, num_labels=len(clases), id2label=dict(enumerate(clases)), label2id={c: i for i, c in enumerate(clases)})
    if forma == "cabeza":
        for nombre, p in red.named_parameters():
            p.requires_grad = nombre.startswith("classifier")
    elif forma == "lora":
        red = get_peft_model(red, LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.1))
    entrenables = sum(p.numel() for p in red.parameters() if p.requires_grad)
    total = sum(p.numel() for p in red.parameters())
    hay_val = val_textos is not None
    args = TrainingArguments(output_dir="entrenamiento", per_device_train_batch_size=lote,
                             per_device_eval_batch_size=64, learning_rate=tasa, num_train_epochs=epocas,
                             warmup_steps=0.1, weight_decay=0.01, eval_strategy="epoch" if hay_val else "no",
                             logging_strategy="epoch", save_strategy="no", report_to="none", seed=semilla,
                             fp16=torch.cuda.is_available(), disable_tqdm=not mostrar)
    entrenador = Trainer(
        model=red, args=args, data_collator=DataCollatorWithPadding(tok),
        train_dataset=_ejemplos(tok, textos, largo, clases, etiquetas),
        eval_dataset=_ejemplos(tok, val_textos, largo, clases, val_etiquetas) if hay_val else None,
        compute_metrics=lambda p: {"exactitud": float((p.predictions.argmax(-1) == p.label_ids).mean())})
    inicio = time.time()
    entrenador.train()
    return Ajuste(entrenador=entrenador, tok=tok, clases=clases, largo=largo, segundos=time.time() - inicio,
                  entrenables=entrenables, total=total, modelo=modelo, forma=forma)

def predecir(ajuste, textos):
    """La clase que el modelo ajustado le da a cada texto."""
    salida = ajuste.entrenador.predict(_ejemplos(ajuste.tok, textos, ajuste.largo))
    return np.array(ajuste.clases)[salida.predictions.argmax(-1)]

def dibujar_historia(ajuste, titulo="Cómo fue el entrenamiento"):
    """Las curvas de la sesión 2: pérdida de entrenamiento y de validación, y exactitud de validación, por época."""
    h = ajuste.entrenador.state.log_history
    ent = [(r["epoch"], r["loss"]) for r in h if "loss" in r]
    val = [(r["epoch"], r["eval_loss"], r["eval_exactitud"]) for r in h if "eval_loss" in r]
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3))
    a.plot([e for e, _ in ent], [l for _, l in ent], "o-", color="#808080", label="Entrenamiento")
    if val:
        a.plot([v[0] for v in val], [v[1] for v in val], "o-", color="#CB318E", label="Validación")
        b.plot([v[0] for v in val], [v[2] for v in val], "o-", color="#CB318E")
        for e, _, x in val:
            b.annotate(pct(x), (e, x), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=9)
        b.set_ylabel("Exactitud de validación"); b.set_ylim(min(v[2] for v in val) - 0.1, 1)
    a.set_ylabel("Pérdida"); a.legend(frameon=False)
    for eje in (a, b):
        eje.set_xlabel("Época"); eje.spines[["top", "right"]].set_visible(False); eje.grid(alpha=0.3)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

def millones(n):
    """118_000_000 -> "118 millones"; 12_345 -> "12.345"."""
    return f"{n / 1e6:.1f} millones".replace(".0 ", " ").replace(".", ",") if n >= 1e6 else miles(n)

from IPython.display import display
REGISTRO = "experimentos.csv"

def registrar(**datos):
    """Agrega una fila a tu registro de experimentos (el archivo experimentos.csv): qué probaste y qué salió."""
    fila = pd.DataFrame([{"hora": time.strftime("%H:%M"), **datos}])
    tabla = pd.concat([pd.read_csv(REGISTRO), fila], ignore_index=True) if os.path.exists(REGISTRO) else fila
    tabla.to_csv(REGISTRO, index=False)

def ver_registro():
    """Muestra el registro de experimentos, el más reciente abajo."""
    if not os.path.exists(REGISTRO):
        print("Todavía no hay experimentos en el registro.")
        return
    tabla = pd.read_csv(REGISTRO)
    display(tabla.style.format(precision=1, decimal=",", thousands=".", na_rep="").hide(axis="index"))

In [ ]:
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-small"   # el de la sesión 6
MODELO_AJUSTE = "intfloat/multilingual-e5-small"       # el que ajustamos; prueba también BETO:
                                                       # "dccuchile/bert-base-spanish-wwm-uncased"
PREFIJO = "query: " if "e5" in MODELO_EMBEDDINGS else ""
_e = SentenceTransformer(MODELO_EMBEDDINGS, device="cuda" if GPU else "cpu")

def embeber(textos):
    """Un vector por texto, de largo 1, con el prefijo que pide el modelo. Pasa los textos a minúsculas."""
    return _e.encode([PREFIJO + str(t).lower() for t in textos], batch_size=32, normalize_embeddings=True,
                     show_progress_bar=False)

---
## A1. Texto

### Tus ejemplos

Los mismos de las sesiones anteriores: escríbelos en `EJEMPLOS` o sube `ejemplos.csv` (columnas `entrada` y
`etiqueta`). Sin datos personales.

In [ ]:
EJEMPLOS = [
    ("Escribe aquí un ejemplo real de tu proyecto.", "etiqueta_1"),
    ("Y otro, con otra etiqueta.", "etiqueta_2"),
]

if os.path.exists("ejemplos.csv"):
    tabla = pd.read_csv("ejemplos.csv")
    EJEMPLOS = list(zip(tabla["entrada"].astype(str), tabla["etiqueta"].astype(str)))
mis_textos = not EJEMPLOS[0][0].startswith("Escribe aquí")
if mis_textos:
    textos, etiquetas = [e for e, _ in EJEMPLOS], [str(et) for _, et in EJEMPLOS]
    print(f"{len(textos)} ejemplos. Etiquetas:", pd.Series(etiquetas).value_counts().to_dict())
else:
    print("Camino A1: escribe tus ejemplos en EJEMPLOS (o sube ejemplos.csv) y vuelve a ejecutar.")

### 1. Lo de la sesión 6: embeddings congelados + regresión logística

In [ ]:
if mis_textos:
    pred = embeddings_con_validacion_cruzada(embeber(textos), etiquetas)
    if pred is not None:
        evaluar(etiquetas, pred, "Embeddings congelados + regresión logística")
        e, b = exactitudes(etiquetas, pred)
        registrar(camino="A1", modelo=MODELO_EMBEDDINGS, forma="embeddings congelados", ejemplos=len(textos),
                  exactitud=e, exactitud_balanceada=b)

### 2. Ajuste fino, con la misma validación cruzada

Ajustamos todo el modelo una vez por grupo de la validación cruzada. Con pocos ejemplos, usamos más épocas (10) y una
tasa un poco más grande (`1e-4`) que en la competencia. Cambia `FORMA` a `"lora"` o `"cabeza"` para comparar.

In [ ]:
FORMA = "todo"
TASA = 1e-4
EPOCAS = 10

if mis_textos:
    pred = ajuste_con_validacion_cruzada(textos, etiquetas, MODELO_AJUSTE, tasa=TASA, epocas=EPOCAS, forma=FORMA)
    if pred is not None:
        evaluar(etiquetas, pred, f"Ajuste fino ({FORMA})")
        e, b = exactitudes(etiquetas, pred)
        registrar(camino="A1", modelo=MODELO_AJUSTE, forma=f"ajuste fino: {FORMA}", tasa=TASA, epocas=EPOCAS,
                  ejemplos=len(textos), exactitud=e, exactitud_balanceada=b)

---
## A2. Imagen

Sube `mis_fotos.zip` con una carpeta por etiqueta (por ejemplo, `mis_fotos/sana/…jpg`), sin personas reconocibles,
como en la sesión 6. Comparamos el mismo modelo,
[`facebook/dinov2-small`](https://huggingface.co/facebook/dinov2-small) (Apache 2.0), de dos formas: congelado, con una
regresión logística encima, y ajustado con el `Trainer`, como en la sesión 4. Puedes cambiar `MODELO_IMAGEN` por tu
modelo base.

In [ ]:
MODELO_IMAGEN = "facebook/dinov2-small"
EPOCAS_IMAGEN = 10

mis_fotos = os.path.exists("mis_fotos.zip")
if mis_fotos:
    from PIL import Image
    from transformers import AutoImageProcessor, AutoModel, AutoModelForImageClassification
    with zipfile.ZipFile("mis_fotos.zip") as z:
        z.extractall(".")
    rutas = sorted(os.path.join(r, f) for r, _, fs in os.walk("mis_fotos") for f in fs
                   if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))
    etiquetas_img = np.array([os.path.basename(os.path.dirname(r)) for r in rutas])
    procesador = AutoImageProcessor.from_pretrained(MODELO_IMAGEN)
    pixeles = torch.cat([procesador(images=Image.open(r).convert("RGB"), return_tensors="pt")["pixel_values"]
                         for r in rutas])
    print(f"{len(rutas)} fotos, etiquetas: {dict(Counter(etiquetas_img))}")

    # 1. Congelado + regresión logística (sesión 6)
    cuerpo = AutoModel.from_pretrained(MODELO_IMAGEN).eval()
    with torch.no_grad():
        salidas = [cuerpo(pixel_values=pixeles[i:i + 16]) for i in range(0, len(pixeles), 16)]
    X_fotos = torch.cat([s.pooler_output if getattr(s, "pooler_output", None) is not None
                         else s.last_hidden_state[:, 0] for s in salidas]).numpy()
    pred = embeddings_con_validacion_cruzada(X_fotos / np.linalg.norm(X_fotos, axis=1, keepdims=True), etiquetas_img)
    evaluar(etiquetas_img, pred, "Embeddings de imagen congelados + regresión logística")
    e, b = exactitudes(etiquetas_img, pred)
    registrar(camino="A2", modelo=MODELO_IMAGEN, forma="embeddings congelados", ejemplos=len(rutas),
              exactitud=e, exactitud_balanceada=b)
else:
    print("Camino A2: sube mis_fotos.zip y vuelve a ejecutar.")

In [ ]:
# 2. Ajuste fino de todo el modelo, con la misma validación cruzada (la receta de la sesión 4)
class Fotos(torch.utils.data.Dataset):
    def __init__(self, idx, etiquetas=None):
        self.idx, self.etiquetas = idx, etiquetas
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        x = pixeles[self.idx[i]]
        if self.etiquetas is not None and torch.rand(1) < 0.5:
            x = torch.flip(x, dims=[2])          # volteo como en un espejo, solo al entrenar
        fila = {"pixel_values": x}
        if self.etiquetas is not None:
            fila["labels"] = self.etiquetas[i]
        return fila

if mis_fotos:
    clases_img = sorted(set(etiquetas_img))
    y_img = np.array([clases_img.index(e) for e in etiquetas_img])
    n = _pliegues(etiquetas_img)
    pred = np.empty(len(rutas), dtype=object)
    inicio = time.time()
    for g, (tr, te) in enumerate(StratifiedKFold(n, shuffle=True, random_state=0).split(rutas, etiquetas_img)):
        red = AutoModelForImageClassification.from_pretrained(MODELO_IMAGEN, num_labels=len(clases_img),
                                                              ignore_mismatched_sizes=True)
        args = TrainingArguments(output_dir="entrenamiento_img", per_device_train_batch_size=8, learning_rate=5e-5,
                                 num_train_epochs=EPOCAS_IMAGEN, warmup_steps=0.1, save_strategy="no",
                                 logging_strategy="no", report_to="none", fp16=GPU, disable_tqdm=True)
        entrenador = Trainer(model=red, args=args, train_dataset=Fotos(tr, y_img[tr]))
        entrenador.train()
        pred[te] = np.array(clases_img)[entrenador.predict(Fotos(te)).predictions.argmax(-1)]
        print(f"  grupo {g + 1} de {n} listo ({time.time() - inicio:.0f} s)")
    evaluar(etiquetas_img, pred, "Ajuste fino de todo el modelo de imágenes")
    e, b = exactitudes(etiquetas_img, pred)
    registrar(camino="A2", modelo=MODELO_IMAGEN, forma="ajuste fino: todo", tasa=5e-5, epocas=EPOCAS_IMAGEN,
              ejemplos=len(rutas), exactitud=e, exactitud_balanceada=b)

---
## B. Práctica: los mensajes ciudadanos

Los 30 mensajes de las sesiones anteriores, cinco por tema. Primero, los embeddings congelados de la sesión 6; después,
el ajuste fino con la misma validación cruzada (cinco modelos, uno por grupo). Para el ajuste fino eliges tú la tasa de
aprendizaje (si te atascas, abre la **pista** que está debajo de la celda).

In [ ]:
MENSAJES = [
    ("El semáforo de la 80 con la 33 lleva tres días apagado y se forman unos trancones terribles.", "movilidad"),
    ("El metro iba tan lleno esta mañana que no pude subirme en tres trenes.", "movilidad"),
    ("Los buses de la ruta del barrio dejaron de pasar después de las ocho de la noche.", "movilidad"),
    ("Hay un hueco enorme en la vía principal y ya se han caído dos motociclistas.", "movilidad"),
    ("No hay señalización en la nueva ciclorruta y los carros se parquean encima.", "movilidad"),
    ("Anoche robaron dos celulares en el paradero del parque y no había ningún policía.", "seguridad"),
    ("Pusieron cámaras en la ciclorruta, pero siguen robando bicicletas.", "seguridad"),
    ("En la esquina de la cancha venden droga a plena luz del día.", "seguridad"),
    ("Las luces del puente peatonal están dañadas y da miedo pasar de noche.", "seguridad"),
    ("Unos muchachos están cobrando vacuna a los tenderos de la cuadra.", "seguridad"),
    ("En el centro de salud del barrio no hay citas para medicina general hasta el próximo mes.", "salud"),
    ("Llevo tres horas esperando en urgencias con mi mamá y nadie nos atiende.", "salud"),
    ("La farmacia de la EPS no tiene la insulina desde hace dos semanas.", "salud"),
    ("Hay muchos casos de dengue en el sector y nadie ha venido a fumigar.", "salud"),
    ("Necesitamos una jornada de vacunación para los niños de la vereda.", "salud"),
    ("La escuela de la vereda no tiene profesor de matemáticas desde marzo.", "educación"),
    ("Al colegio no ha llegado el refrigerio de los estudiantes en todo el mes.", "educación"),
    ("Los computadores de la sala de informática del colegio llevan un año dañados.", "educación"),
    ("No hay cupos en el preescolar del barrio y tengo que trabajar.", "educación"),
    ("El techo del salón se cae a pedazos cuando llueve y los niños reciben clase así.", "educación"),
    ("Hace una semana se fue el agua en toda la cuadra y nadie da razón.", "servicios públicos"),
    ("La factura de la luz llegó el doble de lo normal sin ninguna explicación.", "servicios públicos"),
    ("Hay una fuga de gas en el contador del edificio y huele muy fuerte.", "servicios públicos"),
    ("El alcantarillado se rebosa cada vez que llueve y el agua entra a las casas.", "servicios públicos"),
    ("Se cayó un cable de energía en el andén y está echando chispas.", "servicios públicos"),
    ("Están quemando basura al lado de la quebrada y el humo llega a todas las casas.", "medio ambiente"),
    ("Talaron los árboles del parque sin ningún permiso.", "medio ambiente"),
    ("Una fábrica está botando aguas negras al río.", "medio ambiente"),
    ("La calidad del aire está pésima y los niños tienen tos todo el día.", "medio ambiente"),
    ("Dejan escombros y colchones viejos en la orilla de la quebrada.", "medio ambiente"),
]
textos_b, temas_b = [m for m, _ in MENSAJES], [t for _, t in MENSAJES]

pred = embeddings_con_validacion_cruzada(embeber(textos_b), temas_b)
evaluar(temas_b, pred, "Embeddings congelados + regresión logística, 30 mensajes")
e, b = exactitudes(temas_b, pred)
registrar(camino="B", modelo=MODELO_EMBEDDINGS, forma="embeddings congelados", ejemplos=30, exactitud=e,
          exactitud_balanceada=b)

# Completa: la tasa de aprendizaje del ajuste fino. Pequeña, como en el camino A1.
TASA_B = ...

if TASA_B is ...:
    print("Completa TASA_B y vuelve a ejecutar. Si te atascas, abre la pista de abajo.")
else:
    pred = ajuste_con_validacion_cruzada(textos_b, temas_b, MODELO_AJUSTE, tasa=TASA_B, epocas=10)
    evaluar(temas_b, pred, "Ajuste fino, 30 mensajes")
    e, b = exactitudes(temas_b, pred)
    registrar(camino="B", modelo=MODELO_AJUSTE, forma="ajuste fino: todo", tasa=TASA_B, epocas=10, ejemplos=30,
              exactitud=e, exactitud_balanceada=b)

<details><summary>Pista</summary>

El modelo ya aprendió mucho del idioma; con una tasa grande, los primeros pasos borran lo que sabía. Por eso en el
ajuste fino usamos tasas pequeñas, entre `1e-5` y `1e-4` (`1e-4` es 0,0001). Busca la que usa el camino A1, en
`TASA`: con 30 mensajes y 10 épocas funciona bien.

Después, si quieres, prueba una tasa diez veces más grande y mira qué pasa con la exactitud.
</details>

---
## Tu registro y tu tabla de comparación

El registro tiene todo lo que mediste hoy, con lo que cambiaste en cada prueba.

In [ ]:
ver_registro()

Completa la tabla con lo que mediste en todas las sesiones (doble clic aquí). Usa los mismos ejemplos en todas las
filas. En el camino B, las filas anteriores ya las mediste: zero-shot que lee texto y etiqueta (sesión 4) 63 %, bolsa de
palabras (sesión 5) 13 %, fastText (sesión 5) 60 %.

| Modelo | Exactitud | Exactitud balanceada |
|---|---|---|
| Línea base (la clase más frecuente) | | |
| Tu modelo base (sesión 4) | | |
| Bolsa de palabras (sesión 5) | | |
| Zero-shot con embeddings (sesión 6) | | |
| Embeddings congelados + regresión logística (sesión 6 y hoy) | | |
| Ajuste fino (hoy) | | |

**Lo que observé** (responde en dos o tres líneas):

- ¿El ajuste fino le ganó a los embeddings congelados con tus ejemplos? ¿Por cuánto? ¿Podría ser suerte?
- Según la curva de la demostración, ¿cuántos ejemplos más necesitarías para que el ajuste fino valga la pena?
- Si no pudiste ajustar (por tiempo o por falta de GPU), escribe qué no ejecutaste y por qué.

---
## Entrega

Ejecuta las dos celdas siguientes. La primera solo prepara la función; la segunda lee este notebook con sus
resultados y lo envía a la plataforma. Te pide tu clave la primera vez.

Si algo falla, descarga el notebook (*Archivo › Descargar › Descargar .ipynb*) y súbelo en la plataforma, en la
actividad “Entrega: ajuste fino con tu proyecto”.

In [ ]:
# @title Función para entregar el notebook (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")
ACTIVIDAD = "“Entrega: ajuste fino con tu proyecto”"

def _este_notebook():
    """El notebook tal como se ve en Colab, con sus resultados. Fuera de Colab, el archivo de ESTUDIA_NOTEBOOK."""
    if os.environ.get("ESTUDIA_NOTEBOOK"):
        with open(os.environ["ESTUDIA_NOTEBOOK"], encoding="utf-8") as f:
            return json.load(f)
    try:
        from google.colab import _message
        return _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    except Exception:
        return None

def entregar():
    """Envía este notebook, con lo que ejecutaste, a la plataforma del curso."""
    global _CLAVE
    cuaderno = _este_notebook()
    if not cuaderno:
        print("No pude leer el notebook desde aquí. Descárgalo (Archivo › Descargar › Descargar .ipynb) y súbelo "
              f"en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    if not _CLAVE:
        _CLAVE = getpass.getpass(f"Pega tu clave (plataforma › {ACTIVIDAD} › Copiar) y oprime Enter: ").strip()
    req = urllib.request.Request(f"{PLATAFORMA}/api/entregas/s07", data=json.dumps({"notebook": cuaderno}).encode(),
                                 method="POST", headers={"Content-Type": "application/json",
                                                         "Authorization": f"Bearer {_CLAVE}",
                                                         "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""
        print("No se pudo entregar:", res.get("error", f"error {e.code}"))
        print(f"Si no se arregla, descarga el notebook y súbelo en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    modelos = ", ".join(res["modelos"]) or "ninguno"
    print(f"Entrega recibida. Celdas de código ejecutadas: {res['ejecutadas']} de {res['celdas']}. "
          f"Modelos de Hugging Face que encontré: {modelos}.")
    for aviso in res.get("avisos", []):
        print("Ojo:", aviso)

In [ ]:
entregar()